In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from lib.models import RegNetv3
from lib.data.dataloading import load_nursing_5_class
from lib.modules import optimization_loop_multi_class, evaluate_loop
import torch
from torch import nn
import matplotlib.pyplot as plt
import json
import numpy as np
from lib.utils import sample_regnet
from tqdm import tqdm

In [19]:
CONFIG = {
    'WINDOW_SIZE':2001,
    'NURSING_STRIDE': 2001 // 16,
    'BATCH_SIZE': 512,
    'LEARNING_RATE': 3e-4,
    'TEST_SIZE': 0.25,
    'DEVICE': 'cuda:0',
    'DEPTHI': [1, 1, 2, 2],
    'WIDTHI': [16, 56, 120, 248],
}

In [42]:
nurses = list(set(range(11,71)) - set([37, 46, 70, 39, 22, 13]))
len(nurses)

54

In [21]:
for n in [10,20,30,40,50]:
    _, nursing_trainloader = load_nursing_5_class(
        nurses=list(range(11,11+n)),
        winsize=CONFIG['WINDOW_SIZE'],
        test_size=1,
        batch_size=CONFIG['BATCH_SIZE'],
        stride=CONFIG['NURSING_STRIDE']
    )
    _, nursing_testloader = load_nursing_5_class(
        nurses=[37, 46, 70, 39, 22, 13],
        winsize=CONFIG['WINDOW_SIZE'],
        test_size=1,
        batch_size=CONFIG['BATCH_SIZE'],
        stride=CONFIG['NURSING_STRIDE']
    )
    print(len(nursing_trainloader.dataset), len(nursing_testloader.dataset))
    
    while True:
        d,w,_ = sample_regnet()
        CONFIG['DEPTHI'] = d
        CONFIG['WIDTHI'] = w
        params = sum([p.numel() for p in RegNetv3(CONFIG=CONFIG).parameters()])
        if params < 3_000_000:
            break
    model = RegNetv3(CONFIG=CONFIG).to(CONFIG['DEVICE'])
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['LEARNING_RATE'])

    outdir = f"dev/dataaug/n={n}"
    optimization_loop_multi_class(
        model,
        nursing_trainloader,
        nursing_testloader,
        criterion,
        optimizer,
        epochs=100,
        patience=10,
        device=CONFIG['DEVICE'],
        outdir=outdir,
        writer=outdir,
        label=n,
        config=CONFIG
    )


11364 8480


10: Epoch 64: Train Loss: 0.4358: Dev Loss: 0.85742, Dev F1: 0.62755:  64%|██████▍   | 64/100 [00:56<00:31,  1.13it/s] 


Early stopping at epoch 64
25160 8480


20: Epoch 52: Train Loss: 0.52268: Dev Loss: 0.70477, Dev F1: 0.69907:  52%|█████▏    | 52/100 [01:12<01:07,  1.40s/it]


Early stopping at epoch 52
38456 8480


30: Epoch 56: Train Loss: 0.50959: Dev Loss: 0.54354, Dev F1: 0.76937:  57%|█████▋    | 57/100 [01:48<01:21,  1.90s/it]


KeyboardInterrupt: 